### Lab 9.2 Guess the Missing Letter

In this lab you will set up a Transformer model to guess missing vowels in text.

The data is from the Brown corpus, this time with the vowels masked out:

Input:

    th. f.lt.n c..nty gr.nd j.ry s..d fr.d.y .n .nv.st.g.t..n .f .tl.nt.s r.c.nt pr.m.ry .l.ct..n pr.d.c.d  n. .v.d.nc.  th.t .ny .rr.g.l.r.t..s t..k pl.c. 

Labels:

    ..e..u..o...ou......a....u....ai....i.a..a..i..e..i.a.io..o..a..a..a...e.e.....i.a...e.e..io....o.u.e....o.e.i.e..e....a..a...i..e.u.a.i.ie...oo....a.e.




In [ ]:
import numpy as np
import sklearn
import torch
import os
import pandas as pd
import tqdm

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
import torchmetrics

import pickle

In [ ]:
!wget -nc -q -O brown_corpus2.zip "https://www.dropbox.com/scl/fi/h5tqx5po5m33egd4xgf4h/brown_corpus2.zip?rlkey=y3gw457gs01gybmlt0u78l1fb&dl=1"
!unzip -qq -n brown_corpus2.zip

This time the text has been tokenized by character.

In [ ]:
tokenized_text = pickle.load(open('brown_corpus2/tokens.pkl','rb'))

In [ ]:
labels = pickle.load(open('brown_corpus2/labels.pkl','rb'))

The vocabulary is `. bcdfghjklmnpqrstvwxyz`

The `.` indicates the mask.

In [ ]:
vocab_size = int(np.max([np.max(t) for t in tokenized_text])+1)
vocab_size

The labels are `.aeiou`

In [ ]:
num_labels = int(np.max([np.max(l) for l in labels])+1)
num_labels

This time we will use a maximum sequence length of 100.  To speed up training, we will truncate the test sequences to the first 100 characters, so it doesn't take so long to compute the test accuracy.

In [ ]:
max_seq_len=100

In [ ]:
from sklearn.model_selection import train_test_split
tokenized_text_train, tokenized_text_test, labels_train, labels_test = train_test_split(tokenized_text,labels,test_size=0.1,random_state=42)

In [ ]:
class TokenDataset(Dataset):
  def __init__(self,tokenized_text,labels,max_seq_len=None,randomize=True):
    self.tokenized_text = tokenized_text
    self.labels = labels
    self.max_seq_len = max_seq_len
    self.randomize = randomize
    
  def __len__(self):
    return len(self.tokenized_text)

  def __getitem__(self,idx):
    # get requested text
    token_ids = self.tokenized_text[idx]
    label_ids = self.labels[idx]

    # crop or pad as necessary
    if self.max_seq_len is not None:
      if len(token_ids)>self.max_seq_len:
        if self.randomize:
          # choose random substring
          ind = np.random.randint(len(token_ids)-self.max_seq_len)
        else:
          # choose first substring
          ind = 0
        token_ids = token_ids[ind:ind+self.max_seq_len]
        label_ids = label_ids[ind:ind+self.max_seq_len]
      else:
        # pad to maximum sequence length
        token_ids = [0]*(self.max_seq_len-len(token_ids)) + token_ids
        label_ids = [0]*(self.max_seq_len-len(label_ids)) + label_ids
    
    # return a sequence of token IDs and a label
    return torch.tensor(token_ids), torch.tensor(label_ids).long()


In [ ]:
train_ds = TokenDataset(tokenized_text_train,labels_train,max_seq_len=max_seq_len,randomize=True)
test_ds = TokenDataset(tokenized_text_test,labels_test,max_seq_len=max_seq_len,randomize=False)

batch_size = 32
train_dl = DataLoader(train_ds,shuffle=True,batch_size=batch_size)
test_dl = DataLoader(test_ds,shuffle=False,batch_size=batch_size)

We will use the `ignore_index` flag in the loss function and accuracy metric.

This tells Torch to ignore token zero (the mask token `.`) when computing loss and accuracy.

In [ ]:
device = 'cuda' # or mps

In [ ]:
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

accuracy_metric = torchmetrics.classification.Accuracy(task="multiclass", num_classes=num_labels, ignore_index=0)
accuracy_metric.to(device)

In [ ]:
def train_model(model,num_epochs=10,lr=1e-2):
  opt = torch.optim.Adam(model.parameters(),lr=lr)

  for epoch in range(num_epochs):
    model.train()
    for x_batch, y_batch in tqdm.tqdm(train_dl):
      opt.zero_grad()

      x_batch = x_batch.to(device)
      y_batch = y_batch.to(device)

      y_pred = model(x_batch)
      loss = loss_fn(torch.flatten(y_pred,0,1),torch.flatten(y_batch,0,1))

      loss.backward()

      opt.step()

    model.eval()

    accuracy_metric.reset()
    for x_batch, y_batch in tqdm.tqdm(test_dl):
      x_batch = x_batch.to(device)
      y_batch = y_batch.to(device)
      y_pred = model(x_batch)
      accuracy_metric(torch.flatten(y_pred,0,1),torch.flatten(y_batch,0,1))

    acc = accuracy_metric.compute().item()

    print(f'epoch {epoch}: {acc}')

## Exercises


1. Calculate the rate that each vowel appears in the test set.

*Note: the vowels `a,e,i,o,u` are numbered `1,2,3,4,5` in `labels_test`.

In [ ]:
# YOUR CODE HERE

2. Train an `Embedding` model by itself (with no other modules) for this task.  You should get low test accuracy (about 32%).  Why is the accuracy so low?

*Note: one epoch is sufficient to train the Embedding model.*

In [ ]:
# YOUR CODE HERE

3. This code will retrieve the parameters of the `Embedding`.  

In [ ]:
p = next(iter(model.parameters()))

Apply softmax to the first row of the embedding matrix.  It should appear very similar to the vowel distribution.  Why?

In [ ]:
# YOUR CODE HERE

4. Here is the complete code for a Transformer decoder model.   Train the model on this task.  You should get much higher accuracy (around 75% or higher depending on how long you train for).

*Note: I used an embedding size of 32, 4 attention heads, and 3 attention blocks.*

In [ ]:
class AttentionHead(nn.Module):
    def __init__(self,d_model,d_k):
        super().__init__()
        # create linear projections WQ, WK, WV
        self.WQ = nn.Linear(d_model,d_k)
        self.WK = nn.Linear(d_model,d_k)
        self.WV = nn.Linear(d_model,d_k)

    def forward(self,Q,K,V):
        """ Compute attention head.

            Project the input to queries, keys, and values, and then apply attention.
            Arguments:
                Q: queries [B,L,d_model]
                K: keys    [B,S,d_model]
                V: values  [B,L,d_model]
            Output:
                Context vectors [B,L,d_k]
        """
        # apply linear projections to queries, keys, and values followed by attention
        return F.scaled_dot_product_attention(self.WQ(Q),self.WK(K),self.WV(V))

class MultiHeadAttention(nn.Module):
    def __init__(self,d_model=512,num_heads=8):
        super().__init__()
        d_k = d_model // num_heads
        self.heads = nn.ModuleList([AttentionHead(d_model,d_k) for head in range(num_heads)])
        self.W = nn.Linear(d_model,d_model)

    def forward(self,Q,K,V,mask=None):
        """ Compute multi-head attention.

            Applies attention num_heads times, concatenates the results, and applies a final linear projection.
            Arguments:
                Q: queries [B,L,d_model]
                K: keys    [B,S,d_model]
                V: values  [B,L,d_model]
            Output:
               result of multi-head attention [B,L,d_model]
        """
        # compute each attention head and concatenate
        h = torch.cat([head(Q,K,V) for head in self.heads],dim=-1)

        # apply output projection
        return self.W(h)

class SelfAttentionBlock(nn.Module):
    def __init__(self,d_model=512,num_heads=8,d_ff=2048):
        super().__init__()
        self.multi_head_attention = MultiHeadAttention(d_model,num_heads)
        self.ln1 = nn.LayerNorm(d_model)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model,d_ff),
            nn.SiLU(),
            nn.Linear(d_ff,d_model),
        )
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self,x,mask=None):
        """ Compute self attention block.

            Using the "Pre-LN Transformer" design.

            Arguments:
                x: input sequence [B,S,d_model]
            Output:
               result of attention block [B,L,d_model]
        """
        # layer normalization
        x_ln = self.ln1(x)

        # compute multi-head attention
        mha = self.multi_head_attention(x_ln,x_ln,x_ln)

        # residual connection
        x = mha + x

        # layer normalization
        x_ln2 = self.ln2(x)

        # compute feed-forward network
        ff = self.feed_forward(x_ln2)

        # residual connection
        x = ff + x

        return x
    
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return x

class TransformerDecoder(nn.Module):
    def __init__(self,input_size,max_seq_len,num_heads=8,num_blocks=6):
        super().__init__()
        self.blocks = nn.ModuleList([SelfAttentionBlock(input_size,num_heads,input_size*num_heads) for b in range(num_blocks)])
        self.pe = PositionalEncoding(input_size,max_seq_len)

    def forward(self,x):
        """ Computes the decoded sequence:

            Add positional embedding to input
            Apply self-attention blocks

            Arguments:
                x: input embedding sequence [B,S,input_size]
            Output:
               sequence predictions [B,S,input_size]
        """
        # apply positional embedding
        x = self.pe(x) # [B,S,input_size]
        
        # apply sequence of self-attention blocks
        for block in self.blocks:
            x = block(x) # [B,S,input_size]
        
        return x


In [ ]:
# YOUR CODE HERE